# Healthcare Analytics using PySpark

## Project Overview

This project analyzes healthcare data using Apache Spark and PySpark.

The analysis focuses on patient demographics, treatment costs, hospital revenue, doctor performance, and department-level insights.

### Objectives

- Analyze patient demographics
- Study treatment costs and revenue trends
- Perform data quality validation
- Evaluate doctor performance
- Generate business insights from healthcare data

### Technologies Used

- Python
- PySpark
- Spark SQL
- Data Analysis

## Data Loading and Spark Configuration

This section initializes the Spark session and loads the healthcare datasets for analysis.

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


# Configure Spark environment
spark = SparkSession.builder \
    .appName("MediCore HealthCare Analytics") \
    .getOrCreate()

# Load healthcare datasets
patients_df = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors_df = spark.read.csv("doctors.csv", header=True, inferSchema=True)
treatments_df = spark.read.csv("treatments.csv", header=True, inferSchema=True)

# Explore dataset structure and sample records
print("=== Patients Schema & Sample ===")
patients_df.printSchema()
patients_df.show(5)

print("=== Doctors Schema & Sample ===")
doctors_df.printSchema()
doctors_df.show(5)

print("=== Treatments Schema & Sample ===")
treatments_df.printSchema()
treatments_df.show(5)

# Dataset size overview
print(f"Total Patients: {patients_df.count()}")
print(f"Total Doctors: {doctors_df.count()}")
print(f"Total Treatments: {treatments_df.count()}")

=== Patients Schema & Sample ===
root
 |-- patient_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- insurance_type: string (nullable = true)
 |-- registration_date: date (nullable = true)

+----------+--------------+---+------+------------+--------------+-----------------+
|patient_id|          name|age|gender|        city|insurance_type|registration_date|
+----------+--------------+---+------+------------+--------------+-----------------+
|      1001|    John Smith| 45|     M|      Boston|       Premium|       2024-01-15|
|      1002|  Mary Johnson| 32|     F|    New York|         Basic|       2024-01-16|
|      1003| Michael Brown| 67|     M|Philadelphia|      Standard|       2024-01-18|
|      1004|Patricia Davis| 29|     F|     Chicago|         Basic|       2024-01-20|
|      1005|  David Wilson| 52|     M| Los Angeles|       Premium|       2024-01-22|

## Patient Demographics Analysis

This section explores patient demographics, age distribution, gender distribution, and city-wise patient trends.

In [3]:
#  patient demographic overview

patients_df.select("name", "age", "gender", "city", "insurance_type").show()

+------------------+---+------+------------+--------------+
|              name|age|gender|        city|insurance_type|
+------------------+---+------+------------+--------------+
|        John Smith| 45|     M|      Boston|       Premium|
|      Mary Johnson| 32|     F|    New York|         Basic|
|     Michael Brown| 67|     M|Philadelphia|      Standard|
|    Patricia Davis| 29|     F|     Chicago|         Basic|
|      David Wilson| 52|     M| Los Angeles|       Premium|
|   Jennifer Garcia| 38|     F|       Miami|      Standard|
|      James Miller| 71|     M|     Seattle|    Government|
|   Linda Rodriguez| 43|     F|      Denver|         Basic|
|   Robert Martinez| 36|     M|      Boston|       Premium|
|Elizabeth Anderson| 59|     F|    New York|      Standard|
|    William Taylor| 48|     M|Philadelphia|         Basic|
|    Barbara Thomas| 64|     F|     Chicago|    Government|
|   Richard Jackson| 33|     M| Los Angeles|       Premium|
|       Susan White| 41|     F|       Mi

In [4]:
# Filter patients over 40 years of age

patients_df.filter(col("age") > 40).show()

+----------+------------------+---+------+------------+--------------+-----------------+
|patient_id|              name|age|gender|        city|insurance_type|registration_date|
+----------+------------------+---+------+------------+--------------+-----------------+
|      1001|        John Smith| 45|     M|      Boston|       Premium|       2024-01-15|
|      1003|     Michael Brown| 67|     M|Philadelphia|      Standard|       2024-01-18|
|      1005|      David Wilson| 52|     M| Los Angeles|       Premium|       2024-01-22|
|      1007|      James Miller| 71|     M|     Seattle|    Government|       2024-01-28|
|      1008|   Linda Rodriguez| 43|     F|      Denver|         Basic|       2024-01-30|
|      1010|Elizabeth Anderson| 59|     F|    New York|      Standard|       2024-02-05|
|      1011|    William Taylor| 48|     M|Philadelphia|         Basic|       2024-02-08|
|      1012|    Barbara Thomas| 64|     F|     Chicago|    Government|       2024-02-10|
|      1014|       Su

In [5]:
#  Analyze patient distribution by gender

patients_df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|     F|   74|
|     M|   76|
+------+-----+



In [6]:
# Identify cities represented in the patient dataset

patients_df.select("city").distinct().show()

+------------+
|        city|
+------------+
|Philadelphia|
| Los Angeles|
|     Chicago|
|     Seattle|
|       Miami|
|    New York|
|      Denver|
|      Boston|
+------------+



## Treatment Cost and Revenue Analysis

This section analyzes treatment costs, treatment duration, and revenue generated from different treatments.

In [7]:
# Calculate revenue generated by each treatment

treatments_revenue_df = treatments_df.withColumn("total_revenue", col("treatment_cost") * col("duration_days"))
treatments_revenue_df.show(10)

+------------+----------+---------+-----------+--------------+--------------+-------------+---------+-------------+
|treatment_id|patient_id|doctor_id| department|treatment_date|treatment_cost|duration_days|   status|total_revenue|
+------------+----------+---------+-----------+--------------+--------------+-------------+---------+-------------+
|        2001|      1001|      301| Cardiology|    2024-01-20|        1500.0|            3|Completed|       4500.0|
|        2002|      1002|      305|  Neurology|    2024-01-21|        2200.0|            5|Completed|      11000.0|
|        2003|      1003|      310|Orthopedics|    2024-01-22|        1800.0|            7|Completed|      12600.0|
|        2004|      1004|      314|  Emergency|    2024-01-23|         850.0|            1|Completed|        850.0|
|        2005|      1005|      319| Pediatrics|    2024-01-24|         650.0|            2|Completed|       1300.0|
|        2006|      1006|      323|   Oncology|    2024-01-25|        45

In [8]:
# Identify high-cost treatments

treatments_df.filter(col("treatment_cost") > 2000).show(10)

+------------+----------+---------+-----------+--------------+--------------+-------------+---------+
|treatment_id|patient_id|doctor_id| department|treatment_date|treatment_cost|duration_days|   status|
+------------+----------+---------+-----------+--------------+--------------+-------------+---------+
|        2002|      1002|      305|  Neurology|    2024-01-21|        2200.0|            5|Completed|
|        2006|      1006|      323|   Oncology|    2024-01-25|        4500.0|           14|Completed|
|        2008|      1008|      331|    Surgery|    2024-01-27|        7200.0|           10|Completed|
|        2011|      1011|      311|Orthopedics|    2024-01-30|        2100.0|            8|Completed|
|        2014|      1014|      324|   Oncology|    2024-02-02|        3800.0|           12|Completed|
|        2016|      1016|      332|    Surgery|    2024-02-04|        6800.0|            9|Completed|
|        2018|      1018|      307|  Neurology|    2024-02-06|        2400.0|     

In [9]:
# Top 10 most expensive treatments

treatments_df.orderBy(col("treatment_cost").desc()).show(10)

+------------+----------+---------+----------+--------------+--------------+-------------+---------+
|treatment_id|patient_id|doctor_id|department|treatment_date|treatment_cost|duration_days|   status|
+------------+----------+---------+----------+--------------+--------------+-------------+---------+
|        2165|      1055|      359|   Surgery|    2024-07-07|        9500.0|           15|Completed|
|        2181|      1095|      359|   Surgery|    2024-07-15|        9300.0|           14|Completed|
|        2157|      1035|      334|   Surgery|    2024-07-03|        9200.0|           14|Completed|
|        2195|      1130|      351|   Surgery|    2024-07-22|        9200.0|           14|Completed|
|        2171|      1070|      351|   Surgery|    2024-07-10|        9100.0|           14|Completed|
|        2189|      1115|      334|   Surgery|    2024-07-19|        9000.0|           13|Completed|
|        2175|      1080|      334|   Surgery|    2024-07-12|        8900.0|           13|C

## Data Quality and Validation

This section focuses on improving dataset quality by removing unnecessary fields, standardizing column names, and validating healthcare records.

In [10]:
# Remove non-essential columns

patients_essential_df = patients_df.drop("registration_date")
patients_essential_df.show(5)

+----------+--------------+---+------+------------+--------------+
|patient_id|          name|age|gender|        city|insurance_type|
+----------+--------------+---+------+------------+--------------+
|      1001|    John Smith| 45|     M|      Boston|       Premium|
|      1002|  Mary Johnson| 32|     F|    New York|         Basic|
|      1003| Michael Brown| 67|     M|Philadelphia|      Standard|
|      1004|Patricia Davis| 29|     F|     Chicago|         Basic|
|      1005|  David Wilson| 52|     M| Los Angeles|       Premium|
+----------+--------------+---+------+------------+--------------+
only showing top 5 rows


In [11]:
# Standardize column names

treatments_renamed_df = treatments_df.withColumnRenamed("treatment_cost", "Cost") \
                                     .withColumnRenamed("duration_days", "Days")
treatments_renamed_df.show(5)

+------------+----------+---------+-----------+--------------+------+----+---------+
|treatment_id|patient_id|doctor_id| department|treatment_date|  Cost|Days|   status|
+------------+----------+---------+-----------+--------------+------+----+---------+
|        2001|      1001|      301| Cardiology|    2024-01-20|1500.0|   3|Completed|
|        2002|      1002|      305|  Neurology|    2024-01-21|2200.0|   5|Completed|
|        2003|      1003|      310|Orthopedics|    2024-01-22|1800.0|   7|Completed|
|        2004|      1004|      314|  Emergency|    2024-01-23| 850.0|   1|Completed|
|        2005|      1005|      319| Pediatrics|    2024-01-24| 650.0|   2|Completed|
+------------+----------+---------+-----------+--------------+------+----+---------+
only showing top 5 rows


In [12]:
# Analyze unique doctors and departments

unique_docs = doctors_df.select("doctor_id").distinct().count()
unique_depts = doctors_df.select("department").distinct().count()

print(f"Total Unique Doctors: {unique_docs}")
print(f"Total Unique Departments: {unique_depts}")

print("\nList of Unique Departments:")
doctors_df.select("department").distinct().show()

Total Unique Doctors: 67
Total Unique Departments: 8

List of Unique Departments:
+-----------+
| department|
+-----------+
|  Neurology|
| Cardiology|
| Pediatrics|
|  Emergency|
|Orthopedics|
|    Surgery|
|   Oncology|
|  Radiology|
+-----------+



## Doctor and Department Performance Analysis

This section analyzes doctor performance, department-level treatment trends, revenue generation, and patient-doctor relationships using advanced PySpark operations.

In [13]:
# Combine treatment and doctor datasets

joined_df = treatments_df.join(doctors_df, "doctor_id")
joined_df.show(5)






+---------+------------+----------+-----------+--------------+--------------+-------------+---------+------------+-----------+----------------+-----------+------------+
|doctor_id|treatment_id|patient_id| department|treatment_date|treatment_cost|duration_days|   status| doctor_name| department|experience_years|hourly_rate|        city|
+---------+------------+----------+-----------+--------------+--------------+-------------+---------+------------+-----------+----------------+-----------+------------+
|      301|        2001|      1001| Cardiology|    2024-01-20|        1500.0|            3|Completed|Dr. Anderson| Cardiology|              15|      280.5|      Boston|
|      305|        2002|      1002|  Neurology|    2024-01-21|        2200.0|            5|Completed|   Dr. Davis|  Neurology|              18|      312.0| Los Angeles|
|      310|        2003|      1003|Orthopedics|    2024-01-22|        1800.0|            7|Completed|  Dr. Thomas|Orthopedics|               9|      215.7|

In [14]:
# Remove duplicate department column after join


joined_df = joined_df.drop(doctors_df["department"])
joined_df.show(5)

+---------+------------+----------+-----------+--------------+--------------+-------------+---------+------------+----------------+-----------+------------+
|doctor_id|treatment_id|patient_id| department|treatment_date|treatment_cost|duration_days|   status| doctor_name|experience_years|hourly_rate|        city|
+---------+------------+----------+-----------+--------------+--------------+-------------+---------+------------+----------------+-----------+------------+
|      301|        2001|      1001| Cardiology|    2024-01-20|        1500.0|            3|Completed|Dr. Anderson|              15|      280.5|      Boston|
|      305|        2002|      1002|  Neurology|    2024-01-21|        2200.0|            5|Completed|   Dr. Davis|              18|      312.0| Los Angeles|
|      310|        2003|      1003|Orthopedics|    2024-01-22|        1800.0|            7|Completed|  Dr. Thomas|               9|      215.7|    New York|
|      314|        2004|      1004|  Emergency|    2024-01

In [15]:
# Department-wise treatment and revenue analysis 

department_summary_df = joined_df.groupBy("department").agg(
    count("treatment_id").alias("total_treatments"),
    avg("treatment_cost").alias("avg_treatment_cost"),
    sum(col("treatment_cost") * col("duration_days")).alias("total_revenue"),
    count_distinct("doctor_id").alias("unique_doctors_count")
)
department_summary_df.show()

+-----------+----------------+------------------+-------------+--------------------+
| department|total_treatments|avg_treatment_cost|total_revenue|unique_doctors_count|
+-----------+----------------+------------------+-------------+--------------------+
|  Neurology|              48|            2177.5|     654650.0|                   8|
| Pediatrics|              47| 759.7872340425532|     120110.0|                   8|
| Cardiology|              48|1699.5833333333333|     276870.0|                   8|
|  Emergency|              47| 976.1702127659574|      79680.0|                   8|
|Orthopedics|              47| 2121.063829787234|     701050.0|                   8|
|    Surgery|              58| 7672.413793103448|    4686600.0|                   8|
|   Oncology|              60| 4588.333333333333|    4309100.0|                   8|
|  Radiology|              45|             746.0|      44070.0|                   8|
+-----------+----------------+------------------+-------------+--

In [16]:
# Rank departments by total revenue

sorted_dept_summary_df = department_summary_df.orderBy(col("total_revenue").desc())
sorted_dept_summary_df.show()

+-----------+----------------+------------------+-------------+--------------------+
| department|total_treatments|avg_treatment_cost|total_revenue|unique_doctors_count|
+-----------+----------------+------------------+-------------+--------------------+
|    Surgery|              58| 7672.413793103448|    4686600.0|                   8|
|   Oncology|              60| 4588.333333333333|    4309100.0|                   8|
|Orthopedics|              47| 2121.063829787234|     701050.0|                   8|
|  Neurology|              48|            2177.5|     654650.0|                   8|
| Cardiology|              48|1699.5833333333333|     276870.0|                   8|
| Pediatrics|              47| 759.7872340425532|     120110.0|                   8|
|  Emergency|              47| 976.1702127659574|      79680.0|                   8|
|  Radiology|              45|             746.0|      44070.0|                   8|
+-----------+----------------+------------------+-------------+--

In [17]:
# Analyze city-wise revenue contribution

final_joined_df = joined_df.join(patients_df, on="patient_id", how="inner")

city_revenue_df = final_joined_df.withColumn("revenue", col("treatment_cost") * col("duration_days")) \
    .groupBy(patients_df["city"].alias("patient_city")) \
    .agg(sum("revenue").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

city_revenue_df.show()

+------------+-------------+
|patient_city|total_revenue|
+------------+-------------+
|     Chicago|    2492750.0|
|       Miami|    1972800.0|
|    New York|    1743350.0|
|      Denver|    1213710.0|
|     Seattle|    1017110.0|
|      Boston|     878200.0|
|Philadelphia|     826480.0|
| Los Angeles|     727730.0|
+------------+-------------+



# Project Conclusion

## Key Findings

- Patient demographics were analyzed across multiple cities.
- Treatment costs and revenue patterns were identified.
- Data quality validation improved dataset consistency.
- Department-wise performance and revenue were evaluated.
- City-wise revenue contribution was analyzed.

## Technologies Used

- Python
- PySpark
- Spark SQL
- Data Analysis

## Skills Demonstrated

- Data Cleaning
- Data Transformation
- Aggregations
- Joins
- Revenue Analysis
- Healthcare Analytics